# Phase 3 — Leakage-Safe Preprocessing Pipeline

This notebook develops and validates the preprocessing strategy used by
the ForgeMind APS classification models.

All preprocessing decisions are fitted using training data only.
The official test set is not used for preprocessing decisions or model selection.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup the project root path first
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "dataset"

# 2. Standard sklearn imports
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer


In [6]:
# Now we import both the pipeline preprocessor and the cross-validation helper
from src.preprocessing import build_linear_preprocessor, make_cv_splitter

# Test instantiating them
linear_preprocessor = build_linear_preprocessor()
cv = make_cv_splitter()


In [19]:
from src.data_loader import load_aps_data
train_path = DATA_DIR / "aps_failure_training_set.csv"

X,y = load_aps_data(train_path)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (60000, 170)
y shape: (60000,)
class
0    59000
1     1000
Name: count, dtype: int64


In [20]:
X_train, X_valid, y_train, y_valid = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)

print("Developmentshape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("Full positive rate:", y.mean())
print("Development positive rate:", y_train.mean())
print("Validation positive rate:", y_valid.mean())

Developmentshape: (48000, 170)
Validation shape: (12000, 170)
Full positive rate: 0.016666666666666666
Development positive rate: 0.016666666666666666
Validation positive rate: 0.016666666666666666


In [11]:
linear_preprocessor = build_linear_preprocessor()

In [22]:
X_train_processed = (
    linear_preprocessor.fit_transform(
        X_train,
        y_train
    )
)

X_valid_processed = (
    linear_preprocessor.transform(
        X_valid
    )
)

In [23]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_valid_processed.shape)

Original training shape: (48000, 170)
Processed training shape: (48000, 337)
Processed testing shape: (12000, 337)


Checking for missing values

In [24]:
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Testing missing values:", np.isnan(X_valid_processed).sum())

Training missing values: 0
Testing missing values: 0


In [25]:
assert(X_train_processed.shape[1]==X_valid_processed.shape[1])
assert not np.isnan(X_train_processed).any()

assert not np.isnan(X_valid_processed).any()

print("Linear preprocessing checks passed")

Linear preprocessing checks passed


In [28]:
for fold, (train_index, valid_index) in enumerate(cv.split(X,y), start=1):
    fold_y = y.iloc[valid_index]

    print(f"Fold {fold}:"
    f"rows = {len(valid_index)}, "
    f"Positive rate = {fold_y.mean():.4%}")

Fold 1:rows = 12000, Positive rate = 1.6667%
Fold 2:rows = 12000, Positive rate = 1.6667%
Fold 3:rows = 12000, Positive rate = 1.6667%
Fold 4:rows = 12000, Positive rate = 1.6667%
Fold 5:rows = 12000, Positive rate = 1.6667%


## Phase 3 Conclusions

- Reusable APS data loading was implemented in `src/data_loader.py`.
- The target was encoded as `neg → 0` and `pos → 1`.
- The constant feature `cd_000` is removed inside the pipeline.
- Missing values are replaced using median imputation.
- Missingness indicators preserve the information contained in missing-value patterns.
- RobustScaler handles the extreme differences in feature scale.
- Development and validation preprocessing produced the same 337 features.
- No missing values remained after linear preprocessing.
- Five-fold stratified cross-validation preserved the 1.67% positive rate.
- Preprocessing will remain inside each model pipeline to prevent leakage.
- The official test set remains reserved for final evaluation.